In [1]:
import os
from typing import TypedDict
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:


load_dotenv(dotenv_path=".env", override=True)

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
assert api_key, "API key not found! Check your .env file."

primary_model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key=api_key,
    temperature=0.3,
    max_retries=2,
)

fallback_model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    api_key=api_key,
    temperature=0.3,
    max_retries=2,
)

lite_fallback = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    api_key=api_key,
    temperature=0.3,
    max_retries=2,
)

model = primary_model.with_fallbacks([fallback_model, lite_fallback])

In [12]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str

In [13]:
def create_outline(state: BlogState) -> dict:
    title = state["title"]
    prompt = f"Generate a detailed outline for a blog on the topic - {title}"
    outline = model.invoke(prompt).text
    return {"outline": outline}

In [14]:
def create_blog(state: BlogState) -> dict:
    title = state["title"]
    outline = state["outline"]
    prompt = f"Write a detailed blog on the title - {title} using the following outline \n {outline}"
    content = model.invoke(prompt).text
    return {"content": content}

In [15]:
graph = StateGraph(BlogState)

# nodes
graph.add_node("create_outline", create_outline)
graph.add_node("create_blog", create_blog)

# edges
graph.add_edge(START, "create_outline")
graph.add_edge("create_outline", "create_blog")
graph.add_edge("create_blog", END)

workflow = graph.compile()

In [ ]:
initial_state = {"title": "Rise of AI in Bangladesh"}
final_state = workflow.invoke(initial_state)
print(final_state)

In [ ]:
print(final_state["outline"])
print(final_state["content"])